# LELA60331 Week 7 Seminar: NumPy, and from two classes to many

In Weeks 4 and 5 we built regression models with plain Python lists and loops. That works, but it is long-winded and slow. Most machine-learning code instead uses **NumPy**, a Python library for doing maths on whole lists and tables of numbers at once. This week we:

1. learn the basics of NumPy
2. rewrite last week's Yelp sentiment classifier with NumPy
3. start looking at **multiclass** classification, where there are more than two possible labels

| Part | Topic |
|---|---|
| 1 | A short NumPy tutorial |
| 2 | Logistic regression with NumPy |
| 3 | Multiclass classification and softmax |

**Lectures:** this sheet goes with this week's lecture, *Neural Networks 1*. Part 3 uses its slides on multiclass logistic regression and the softmax function.

---
# Part 1 · A short NumPy tutorial

## Arrays

NumPy's main type is the **array**: like a list, but all its elements are numbers, and arithmetic works **element by element** on the whole array at once. We import NumPy under the short name `np`:

In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print(a + b)          # element by element: [11 22 33]
print(a * b)          # [10 40 90]
print(a * 2, a + 1)   # a single number is applied to every element
print(np.exp(a))      # NumPy's maths functions also work on every element
print(a.sum(), a.mean(), len(a))

Compare this with lists, where `+` joins two lists and `*` repeats a list:

In [ ]:
print([1, 2, 3] + [10, 20, 30])
print([1, 2, 3] * 2)

The **dot product** (multiply element by element, then add up) is `a.dot(b)`. In Week 4 we wrote our own `dot` function with a loop. NumPy's is built in, and much faster.

In [ ]:
print(a.dot(b))       # 1*10 + 2*20 + 3*30 = 140

### 🏋️ Exercise 1 — Week 4's regression in NumPy

In Week 4 Activity 1 we used list comprehensions to compute predictions and errors for every data point. With NumPy arrays each step is a single, short expression with no loop at all. Complete:

1. `y_est`: `bias + x * weight` for every `x`
2. `errors`: `y_est` minus `y`
3. `loss`: the mean squared error, `errors.dot(errors)` divided by the number of data points
4. `dw`: `x.dot(errors)` divided by the number of data points

In [ ]:
x = np.array([1.00, 1.25, 1.50, 1.75, 2.00, 2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00])
y = np.array([33, 49, 41, 54, 52, 45, 36, 58, 45, 69, 55, 56, 68])
bias = 10
weight = 15
num_samples = len(y)

y_est = ???
errors = ???
loss = ???
dw = ???
print(loss, dw)       # you should get 119.625 and -1.875

## Matrices

A two-dimensional array is a **matrix**: a table of numbers with rows and columns. You make one from a list of lists (LELA60341 Week 4). Its `.shape` gives the number of rows and columns, and you index it with `[row, column]`. A `:` on its own means "everything", so `m[:, 1]` is the whole of column 1.

In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])

print(m.shape)        # 2 rows, 3 columns
print(m[1, 2])        # row 1, column 2
print(m[0])           # row 0
print(m[:, 1])        # column 1
print(m.T)            # the transpose: rows become columns
print(m.sum(axis=0))  # add up down each column
print(m.sum(axis=1))  # add up along each row

**Matrix-vector multiplication**, `m.dot(v)`, takes the dot product of **each row** of `m` with the vector `v`, giving one number per row. The number of columns of `m` must equal the length of `v`. This is how we will make predictions for every data point at once: each row of the matrix is one data point, and `v` holds the weights.

In [ ]:
v = np.array([1, 0, 2])
print(m.dot(v))       # [1*1 + 2*0 + 3*2, 4*1 + 5*0 + 6*2] = [7, 16]

### Making arrays

| Code | Result |
|---|---|
| `np.zeros(5)`, `np.zeros((2, 3))` | an array of 0s: 5 long, or 2 rows × 3 columns |
| `np.ones(4)` | an array of 1s |
| `np.random.seed(1)` | (makes the random numbers repeatable, as with `random.seed`) |
| `np.random.rand(3)`, `np.random.rand(2, 3)` | random numbers between 0 and 1 |
| `np.random.normal(mean, sd, size)` | random numbers from a normal distribution |
| `np.random.permutation(10)` | the numbers 0 to 9 in a random order |

In [ ]:
print(np.zeros((2, 3)))
np.random.seed(1)
print(np.random.rand(3))
print(np.random.permutation(10))

### 🏋️ Exercise 2 — Working with a matrix

`scores` holds the scores of 3 students (rows) in 4 tests (columns). Using NumPy (no loops):

1. `score_2_3`: the score of student 2 in test 3
2. `totals`: each student's total score
3. `test_0`: every student's score in test 0
4. `weighted`: each student's weighted score, where the tests count for `test_weights`. *Hint:* matrix-vector multiplication.

In [ ]:
scores = np.array([[7, 5, 9, 6],
                   [4, 8, 6, 10],
                   [9, 9, 7, 3]])
test_weights = np.array([0.1, 0.2, 0.3, 0.4])

score_2_3 = ???
totals = ???
test_0 = ???
weighted = ???
print(score_2_3, totals, test_0, weighted)    # 3 [27 28 28] [7 4 9] [6.8 7.8 6. ]

## Comparisons and selecting elements

Comparing an array with a number gives an array of `True`/`False` values, one per element. You can use such an array to **select** the elements where it is `True`, or add it up to **count** them (`True` counts as 1). To combine two conditions use `&` ("and") with brackets around each condition, because Python's `and` doesn't work element by element.

In [ ]:
q = np.array([0.9, 0.2, 0.7, 0.4])
labels_01 = np.array([1, 0, 0, 1])

print(q > 0.5)                       # [ True False  True False]
print(q[q > 0.5])                    # the elements that are above 0.5
print(np.sum(q > 0.5))               # how many are above 0.5
print((q > 0.5).astype(int))         # True/False as 1/0
print(np.sum((q > 0.5) & (labels_01 == 1)))    # above 0.5 AND labelled 1

Finally, `np.argmax` gives the **position** of the largest element. With `axis=1` it does this for each row of a matrix:

In [ ]:
print(np.argmax(np.array([0.1, 0.7, 0.2])))       # 1
probs = np.array([[0.1, 0.7, 0.2],
                  [0.5, 0.3, 0.2]])
print(np.argmax(probs, axis=1))                   # [1 0]

### 🏋️ Exercise 3 — Predictions and counts

`q` holds a classifier's predicted probabilities that each of 8 reviews is positive, and `y_true` holds their real labels (1 = positive).

1. `y_pred`: 1 where `q` is above 0.5 and 0 elsewhere
2. `accuracy`: the proportion of predictions that are right. *Hint:* `np.mean` of a comparison.
3. `tp`, `fp` and `fn`: the numbers of true positives, false positives and false negatives (Week 5)

In [ ]:
q = np.array([0.95, 0.10, 0.65, 0.40, 0.80, 0.55, 0.30, 0.05])
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0])

y_pred = ???
accuracy = ???
tp = ???
fp = ???
fn = ???
print(y_pred, accuracy, tp, fp, fn)     # [1 0 1 0 1 1 0 0] 0.75 3 1 1

---
# Part 2 · Logistic regression with NumPy

In Week 5 we trained a logistic regression model to classify Yelp reviews as positive or negative, using lists and loops. Now we'll do it with NumPy. Run the next two cells: the first downloads the data, the second prepares it exactly as in Week 5, except that the one-hot matrices `M_train` and `M_test` are now NumPy matrices (one row per review, one column per vocabulary word) and the labels are NumPy arrays of 1s and 0s. It takes a minute or so.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

reviews = []
labels = []
with open("yelp_reviews.txt") as fp:
    lines = fp.readlines()
for line in lines[1:]:                         # skip the header line
    fields = line.rstrip().split("\t")         # [review, label]
    reviews.append(fields[0])
    labels.append(fields[1])

# Tokenise each review by splitting at spaces
tokenized_sents = []
for review in reviews:
    tokenized_sents.append(review.split())

# The 5000 most frequent words
counts = {}
for sent in tokenized_sents:
    for word in sent:
        counts[word] = counts.get(word, 0) + 1
pairs = sorted([(count, word) for word, count in counts.items()], reverse=True)
type_list = [word for count, word in pairs[:5000]]
word_to_index = {}
for j in range(len(type_list)):
    word_to_index[type_list[j]] = j

# Split the reviews: 8000 for training, 2000 for testing
np.random.seed(1)
order = np.random.permutation(len(reviews))    # the numbers 0 to 9999 in a random order
train_ints = order[:8000]
test_ints = order[8000:]

def one_hot(indices):
    """One row per review; column j is 1 if word j occurs in the review."""
    M = np.zeros((len(indices), len(type_list)))
    for row in range(len(indices)):
        for word in tokenized_sents[indices[row]]:
            if word in word_to_index:
                M[row, word_to_index[word]] = 1
    return M

M_train = one_hot(train_ints)
M_test = one_hot(test_ints)
y_train = np.array([int(labels[i] == "positive") for i in train_ints])
y_test = np.array([int(labels[i] == "positive") for i in test_ints])
print(M_train.shape, M_test.shape, y_train.shape)

### 🏋️ Exercise 4 — Training with NumPy

Complete the training loop. It does exactly the same as in Week 5, but each step is now one NumPy expression:

* `z`: `M_train.dot(weights)` plus the bias gives z for every training review at once
* `q`: the sigmoid of `z`, using `np.exp`
* `dw`: `M_train.T.dot(q - y_train)` divided by the number of samples gives the gradient for every weight at once
* `db`: the sum of `q - y_train` divided by the number of samples

How long does it take, compared with Week 5?

In [ ]:
np.random.seed(2)
num_features = M_train.shape[1]
num_samples = M_train.shape[0]
weights = np.random.rand(num_features)
bias = np.random.rand()
n_iters = 500
lr = 0.4
eps = 0.00001
logistic_loss = []

for i in range(n_iters):
    z = ???
    q = ???
    loss = -np.sum(y_train * np.log2(q + eps) + (1 - y_train) * np.log2(1 - q + eps))
    logistic_loss.append(loss)

    dw = ???
    db = ???
    weights = weights - lr * dw
    bias = bias - lr * db

plt.plot(range(1, n_iters), logistic_loss[1:])
plt.xlabel("number of epochs")
plt.ylabel("loss")
plt.show()

### 🏋️ Exercise 5 — Evaluation with NumPy

1. Calculate `q_test`, the predicted probabilities for the test reviews, and `y_test_pred`, the predicted labels (1 or 0).
2. Calculate accuracy, precision and recall on the test set, as in Exercise 3. (Precision = tp / (tp + fp) and recall = tp / (tp + fn).)
3. Do the same for the **training** set. How do the numbers differ? Why?

In [ ]:
q_test = ???
y_test_pred = ???

tp = ???
fp = ???
fn = ???
print("test accuracy:", np.mean(y_test_pred == y_test))
print("test precision:", tp / (tp + fp), " recall:", tp / (tp + fn))

In [ ]:
# 3. precision and recall on the training set

---
# Part 3 · Multiclass classification and softmax

Logistic regression is great when there are two possible labels. But many classification problems have **more than two**. Sentiment, for example, could be **positive**, **negative** or **neutral**.

Here is a simulated data set for that three-class problem. As in Week 4, each text has two features: the log count of positive words and the log count of negative words. Positive texts have many positive words and few negative ones; negative texts the reverse; and neutral texts have either few of both or many of both.

In [ ]:
np.random.seed(10)
positive = np.random.normal((3, 1), 0.5, size=(20, 2))       # 20 texts around (3, 1)
negative = np.random.normal((1, 3), 0.5, size=(20, 2))       # 20 texts around (1, 3)
neutral = np.concatenate([np.random.normal((1, 1), 0.5, size=(10, 2)),
                          np.random.normal((3, 3), 0.5, size=(10, 2))])
x = np.concatenate([positive, negative, neutral])            # 60 rows, 2 columns
labs = np.array([0] * 20 + [1] * 20 + [2] * 20)              # 0 = positive, 1 = negative, 2 = neutral

plt.scatter(x[labs == 0, 0], x[labs == 0, 1], marker="o", s=60, label="positive")
plt.scatter(x[labs == 1, 0], x[labs == 1, 1], marker="*", s=100, label="negative")
plt.scatter(x[labs == 2, 0], x[labs == 2, 1], marker="x", s=60, label="neutral")
plt.xlabel("log count of positive words")
plt.ylabel("log count of negative words")
plt.legend()
plt.show()

Can a single straight line separate the three classes? No: we need a way of scoring **each** class.

### Representing the labels: one-hot vectors

With two classes, each label was a single 0 or 1. With several classes, we represent each label as a **one-hot vector**: a vector of 0s with a single 1 in the position of the correct class.

| class | one-hot vector |
|---|---|
| 0 positive | `[1, 0, 0]` |
| 1 negative | `[0, 1, 0]` |
| 2 neutral | `[0, 0, 1]` |

`np.eye(3)` gives a 3 × 3 matrix with 1s on the diagonal, whose rows are exactly these vectors, so `np.eye(3)[labs]` turns the list of class numbers into a matrix of one-hot rows.

In [ ]:
print(np.eye(3))
y = np.eye(3)[labs]
print(y.shape)
print(y[18:23])          # the last two positive texts and the first three negative ones

### Softmax

In logistic regression we calculated **one** value z for each data point (the dot product of its features with the weights, plus the bias) and turned it into a probability with the sigmoid function.

In **multinomial logistic regression** (also called **softmax regression**), each class has its **own** weights and bias, so we calculate a separate z for each class. The **softmax** function then turns these z values into probabilities that add up to 1. For three classes:

$$p(y=k \mid x) = \frac{e^{z_k}}{e^{z_0} + e^{z_1} + e^{z_2}}$$

In words: exponentiate each z, then divide each one by their total. A class with a bigger z gets a bigger share of the probability.

### 🏋️ Exercise 6 — The softmax function

Complete `softmax(z)`, which takes a NumPy array of z values and returns the array of probabilities. Use `np.exp` and `.sum()`.

In [ ]:
def softmax(z):
    e = ???
    return ???

p = softmax(np.array([2.0, 1.0, 0.1]))
print(p, p.sum())          # [0.659 0.242 0.099] and 1.0

### 🏋️ Exercise 7 — Class probabilities from a fitted model

A fitted model has the weights and biases below: one row of `weights` (and one bias) for each class, with the first weight for the positive-word feature and the second for the negative-word feature.

For each text, calculate the probability of each class. Remember that the features are **log** counts, so use `np.log` on the counts.

a) 10 positive words and 3 negative words
b) 3 positive words and 3 negative words
c) 1 positive word and 6 negative words

*Hint:* `weights.dot(features) + biases` gives all three z values at once.

In [ ]:
#              positive-word weight, negative-word weight
weights = np.array([[ 1.80, -0.75],       # class 0: positive
                    [-0.70,  1.81],       # class 1: negative
                    [ 0.30,  0.30]])      # class 2: neutral
biases = np.array([-0.45, -0.82, 0.50])

def class_probabilities(n_positive, n_negative):
    features = ???
    z = ???
    return softmax(z)

np.set_printoptions(suppress=True, precision=3)    # print without scientific notation
print("a)", class_probabilities(10, 3))
print("b)", class_probabilities(3, 3))
print("c)", class_probabilities(1, 6))

Do the most probable classes make sense?

### 🏋️ Exercise 8 — Predicted classes

The predicted class is the one with the highest probability. Use `np.argmax` to set `predicted` to a list of the predicted class numbers for the three texts above, and `class_names` to turn each into a name.

In [ ]:
class_names = ["positive", "negative", "neutral"]
texts = [(10, 3), (3, 3), (1, 6)]

predicted = ???
print([class_names[k] for k in predicted])

---
## ✏️ Summary

This week you have:

* used NumPy arrays and matrices: element-by-element arithmetic, `dot`, `.T`, `.shape`, `sum` with `axis`, comparisons, selection and `argmax`
* trained and evaluated a logistic regression classifier with NumPy
* represented multiclass labels as **one-hot vectors**, and turned a z value per class into probabilities with **softmax**

Next week we will **train** a softmax classifier.